# Figuras y cifras del capítulo 4 — Curación del conjunto de datos (OE1)

Todo lo que el capítulo cita sale de aquí: nada de números a mano en el `.tex`.
El cuaderno escribe cuatro cosas en `../figures/`:

| Salida | Qué es | Cómo lo usa el `.tex` |
|---|---|---|
| `valores.tex` | macros con cada cifra (`\nKept`, `\pctExperiment`, …) | `\input` en el preámbulo del capítulo |
| `tablas/*.tex` | las filas de cada tabla, sin cabecera ni `\bottomrule` | `\input{…} \\` dentro del `tabular` |
| `data/*.dat` | series para pgfplots | `\addplot table` |
| `*.png` | las figuras que son **imagen o nube de puntos** | `\includegraphics` |

**Qué va en matplotlib y qué va en LaTeX.** Aquí solo se generan las figuras que
rinden una imagen (espectrogramas) o que dibujan cientos de puntos (nubes,
facetas, barras por clase). Los **esquemas** (el orden de las reglas, el reparto
por grabación, el ventaneo, la normalización de la caja, las capas del conjunto)
se dibujan en **TikZ** dentro del `.tex`, y los **histogramas de pocas barras**
(cajas por ventana, hora del día, mes) en **pgfplots** desde los `.dat`: así el
texto de la figura usa la tipografía del documento y no una imagen escalada.

In [1]:
import hashlib
import json
import shutil
import sys
from collections import Counter
from collections.abc import Iterable
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Rectangle
from matplotlib.ticker import NullFormatter
from slugify import slugify

NB_DIR = Path.cwd()
CHAPTER_DIR = NB_DIR.parent
PROJECT_DIR = next(p for p in (NB_DIR, *NB_DIR.parents) if (p / "pyproject.toml").exists())
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))

FIG = CHAPTER_DIR / "figures"
# Ruta con la que el .tex nombra estas figuras (relativa a research/main.tex).
RUTA_FIG = "capitulos/04_oe1_curacion/figures"
TAB = FIG / "tablas"
DAT = FIG / "data"
for path in (FIG, TAB, DAT):
    path.mkdir(parents=True, exist_ok=True)

# Ancho del texto del documento (A4 con márgenes 3 + 2,5 cm): una figura de este ancho se
# imprime a escala 1 y sus letras salen del tamaño que se ve acá.
TEXT_WIDTH_IN = 6.1
DPI = 200
# La paleta de los esquemas TikZ (research/estilos.tex), para que figura y esquema combinen.
AZUL, CELESTE, VERDE, ROSA, AMBAR, GRIS = (
    "#1E40AF",
    "#00B4D8",
    "#22C55E",
    "#FF5FA2",
    "#F59E0B",
    "#6B7280",
)
mpl.rcParams.update(
    {
        "figure.dpi": 100,
        "savefig.dpi": DPI,
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def number(value: float | int, decimals: int = 0) -> str:
    """Cifra a la española: miles con espacio fino y coma decimal."""
    if decimals == 0:
        return f"{round(value):,}".replace(",", "\\,")
    whole, fraction = f"{value:,.{decimals}f}".split(".")
    return whole.replace(",", "\\,") + "," + fraction


def percent(value: float, decimals: int = 1) -> str:
    return number(100 * value, decimals) + "\\,\\%"


def macro_name(key: str) -> str:
    # `n_boxes_train` -> `nBoxesTrain`; LaTeX no admite dígitos ni guiones en un comando.
    words = key.replace("-", "_").split("_")
    name = words[0] + "".join(w.capitalize() for w in words[1:])
    return "".join(c for c in name if c.isalpha())


def write_macros(values: dict[str, str]) -> None:
    lines = [f"\\newcommand{{\\{macro_name(k)}}}{{{v}}}" for k, v in values.items()]
    (FIG / "valores.tex").write_text(
        "% generado por notebook/figuras_04_oe1_curacion.ipynb; no editar\n"
        + "\n".join(lines)
        + "\n"
    )


def write_rows(name: str, rows: Iterable[Iterable[object]]) -> None:
    # Sólo las filas, sin `\\` final: un `\input` que termina en `\\` rompe el `\bottomrule`
    # que le sigue, así que el `.tex` escribe `\input{...} \\`.
    body = " \\\\\n".join(" & ".join(str(cell) for cell in row) for row in rows)
    (TAB / f"{name}.tex").write_text("% generado por el cuaderno; no editar\n" + body + "\n")


def write_data(name: str, frame: pd.DataFrame) -> None:
    frame.to_csv(DAT / f"{name}.dat", sep=" ", index=False)


def save(fig, name: str) -> None:
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight", dpi=DPI)


def scalar(frame: pd.DataFrame, row, column: str) -> float:
    return float(frame[column].to_numpy()[frame.index.get_loc(row)])


VALORES: dict[str, str] = {}
print(f"proyecto: {PROJECT_DIR}\nfiguras:  {FIG}")

proyecto: /home/fcandia/tesis-primate
figuras:  /home/fcandia/tesis-primate/research/capitulos/04_oe1_curacion/figures


## 1. RE1.1 — auditoría del protocolo, regla por regla

Se vuelve a aplicar el protocolo de `src/data/annotations.py` sobre las tablas crudas de
`data/raw/`, contando qué toca cada regla. Es la evidencia de la tabla de reglas y de las
cifras del apartado de RE1.1.

In [2]:
from core.config import PROCESSED_DIR, RAW_DIR, P  # noqa: E402
from data.annotations import (  # noqa: E402
    CALL_SYNONYMS,
    DROP_COLUMNS,
    MANUAL_FIXES,
    MANUAL_SYNONYMS,
    MAX_FREQ_HZ,
    MIN_DURATION_S,
    NOISE,
    SPECIES_CODES,
    list_files,
    recording_stem,
    species_of,
)
from data.annotations import cleaned as cleaned_column  # noqa: E402
from data.species import CALL_TYPES, VALID_PAIRS  # noqa: E402

RAW_SPECIES, RAW_CALL = "species", "call_type"


def recording_of(table: Path) -> Path | None:
    for suffix in (".wav", ".WAV"):
        candidate = table.with_name(recording_stem(table) + suffix)
        if candidate.is_file():
            return candidate
    return None


# Las carpetas que no son una especie del corpus (aves, sólo para preentrenar) quedan fuera.
tables = [p for p in list_files(RAW_DIR, ".txt") if species_of(p) in SPECIES_CODES]
recordings_raw = [
    p
    for p in list_files(RAW_DIR, ".wav") + list_files(RAW_DIR, ".WAV")
    if species_of(p) in SPECIES_CODES
]
paired = {t: audio for t in tables if (audio := recording_of(t)) is not None}
orphans = [t for t in tables if t not in paired]
print(f"{len(recordings_raw)} grabaciones · {len(tables)} tablas · {len(paired)} emparejadas")

3356 grabaciones · 2835 tablas · 2823 emparejadas


In [3]:
def audit(table: Path, species: str) -> tuple[pd.DataFrame, pd.DataFrame, Counter]:
    """Aplica las reglas en el orden de `clean_annotations` y cuenta qué toca cada una."""
    raw = pd.read_csv(table, sep="\t")
    hits: Counter = Counter(filas=len(raw))
    hits.update({f"col:{c}": 1 for c in raw.columns})
    df = raw.copy()
    df.columns = [cleaned_column(c) for c in df.columns]
    manual = df[[RAW_SPECIES, RAW_CALL]].copy()
    df = df.drop(columns=DROP_COLUMNS, errors="ignore")

    typed = df[RAW_CALL].map(lambda v: v if isinstance(v, str) else None)
    slug = typed.map(lambda v: slugify(v, separator="_") or None if isinstance(v, str) else None)
    # Más allá de la mayúscula: espacios, signos y separadores que la slugificación absorbe.
    hits["tipografía"] = int((typed.str.strip().str.lower().fillna("") != slug.fillna("")).sum())
    synonyms = MANUAL_SYNONYMS | CALL_SYNONYMS.get(species, {})
    mapped = slug.replace(synonyms)
    hits["sinónimo"] = int((slug.fillna("") != mapped.fillna("")).sum())
    hits["vacío"] = int(mapped.isna().sum())
    hits["ruido"] = int(mapped.eq(NOISE).sum())
    df[RAW_CALL] = mapped

    written = df[RAW_SPECIES].map(lambda v: v.strip().lower() if isinstance(v, str) else "")
    other_species = written.ne("") & written.ne(species)
    hits["especie vacía"] = int(written.eq("").sum())
    hits["especie≠carpeta"] = int(other_species.sum())
    df["written_species"] = written
    df[RAW_SPECIES] = species
    df = df.loc[df[RAW_CALL].ne(NOISE)]

    for (bad_sp, bad_ct), (sp, ct) in MANUAL_FIXES.items():
        wrong = df[RAW_SPECIES].eq(bad_sp) & df[RAW_CALL].eq(bad_ct)
        hits["par imposible"] += int(wrong.sum())
        df.loc[wrong, [RAW_SPECIES, RAW_CALL]] = [sp, ct]

    pairs = pd.Series(list(zip(df[RAW_SPECIES], df[RAW_CALL], strict=True)), index=df.index)
    out_of_vocabulary = ~pairs.isin(VALID_PAIRS)
    hits["fuera de vocabulario"] = int((out_of_vocabulary & df[RAW_CALL].notna()).sum())
    df["requires_review"] = out_of_vocabulary | other_species[df.index]
    hits["en revisión"] = int(df["requires_review"].sum())

    hits["sobre Nyquist"] = int((df["high_freq_hz"] > MAX_FREQ_HZ).sum())
    df["high_freq_hz"] = df["high_freq_hz"].clip(upper=MAX_FREQ_HZ)
    df["duration_s"] = df["end_time_s"] - df["begin_time_s"]
    df["bandwidth_hz"] = df["high_freq_hz"] - df["low_freq_hz"]
    degenerate = (df["duration_s"] < MIN_DURATION_S) | (df["bandwidth_hz"] <= 0)
    hits["degenerada"] = int(degenerate.sum())
    df = df.loc[~degenerate]
    hits["conservadas"] = len(df)
    return df.assign(audio_path=str(recording_of(table)), source=str(table)), manual, hits


frames, manuals, counts, failed = [], [], Counter(), []
for table, audio in paired.items():
    try:
        frame, manual, hits = audit(table, species_of(audio))
    except Exception as exc:  # una tabla ilegible no detiene la auditoría, igual que el pipeline
        failed.append(f"{table.relative_to(RAW_DIR)}: {exc}")
        continue
    frames.append(frame)
    manuals.append(manual)
    counts.update(hits)
audited = pd.concat(frames, ignore_index=True)
raw_manual = pd.concat(manuals, ignore_index=True)
print(f"{counts['filas']} filas crudas · {len(failed)} tablas ilegibles")
failed

20082 filas crudas · 4 tablas ilegibles


['howler_monkey__AS/20240120_065845.txt: No columns to parse from file',
 "night_monkey__AA/20240609_174227.txt: 'utf-8' codec can't decode byte 0x80 in position 37: invalid start byte",
 "night_monkey__AA/20240609_174409.txt: 'utf-8' codec can't decode byte 0x80 in position 37: invalid start byte",
 'peruvian_spider_monkey__AC/20240409_064044.txt: No columns to parse from file']

In [4]:
# Copias byte-idénticas del mismo wav en varias carpetas de especie: una sola grabación con la
# unión de sus anotaciones (lo que hace `unify_copies` al cargar).
def signature(path: Path) -> tuple[int, str]:
    with open(path, "rb") as handle:
        return path.stat().st_size, hashlib.md5(handle.read(1024 * 1024)).hexdigest()


signatures = {p: signature(Path(p)) for p in audited["audio_path"].unique()}
first_with = {}
canonical = {}
for path, sig in sorted(signatures.items()):
    canonical[path] = first_with.setdefault(sig, path)
groups = pd.Series(canonical).groupby(lambda p: canonical[p]).size()
unified = audited.assign(audio_path=audited["audio_path"].map(canonical))
duplicated = unified.duplicated(subset=[c for c in unified.columns if c != "source"])
n_duplicated_rows = int(duplicated.sum())
kept = unified.loc[~duplicated]
n_copies = int((groups - 1).clip(lower=0).sum())
print(
    f"{int((groups > 1).sum())} grabaciones con copia · {n_copies} copias unidas · "
    f"{len(groups)} grabaciones tras unir · {n_duplicated_rows} anotaciones repetidas"
)

120 grabaciones con copia · 128 copias unidas · 2691 grabaciones tras unir · 0 anotaciones repetidas


In [5]:
# Tabla: qué toca cada regla, en el orden en que se aplican.
n_review = int(kept["requires_review"].sum())
reglas = [
    ("Tablas sin grabación homónima", len(orphans), "se omiten"),
    ("Tablas ilegibles", len(failed), "se omiten y se registran"),
    ("Filas crudas en tablas con audio", counts["filas"], "punto de partida"),
    ("Tipo de llamada con espacios o signos de más", counts["tipografía"], "se normaliza"),
    ("Tipo de llamada por sinónimo o nombre legible", counts["sinónimo"], "mapa explícito"),
    ("Especie vacía", counts["especie vacía"], "la de la carpeta"),
    (
        "Especie escrita distinta de la carpeta",
        counts["especie≠carpeta"],
        "la de la carpeta $+$ \\code{requires\\_review}",
    ),
    ("Tipo de llamada vacío", counts["vacío"], "\\code{requires\\_review}"),
    ("Tipo de llamada \\code{noise}", counts["ruido"], "se descarta"),
    ("Par imposible corregido", counts["par imposible"], "\\code{aa/hc} $\\to$ \\code{aa/hm}"),
    (
        "Frecuencia superior por encima de Nyquist",
        counts["sobre Nyquist"],
        "se recorta a 22\\,050\\,Hz",
    ),
    ("Caja degenerada ($<10$\\,ms o sin banda)", counts["degenerada"], "se descarta"),
    ("Par fuera de vocabulario", counts["fuera de vocabulario"], "\\code{requires\\_review}"),
    ("Copias del mismo audio en otra carpeta", n_copies, "se unen a su original"),
    ("Anotaciones repetidas entre copias", n_duplicated_rows, "se descartan"),
    ("Anotaciones conservadas", len(kept), "$\\to$ \\file{data/cleaned/}"),
    (
        "De ellas, marcadas \\code{requires\\_review}",
        n_review,
        "fuera del experimento (RE1.2)",
    ),
]
write_rows("reglas", [(r, number(n), how) for r, n, how in reglas])
pd.DataFrame(reglas, columns=["regla", "filas", "tratamiento"])

,regla,filas,tratamiento
0,Tablas sin grabación homónima,12,se omiten
1,Tablas ilegibles,4,se omiten y se registran
2,Filas crudas en tablas con audio,20082,punto de partida
3,Tipo de llamada con espacios o signos de más,52,se normaliza
4,Tipo de llamada por sinónimo o nombre legible,105,mapa explícito
5,Especie vacía,879,la de la carpeta
6,Especie escrita distinta de la carpeta,129,la de la carpeta $+$ \code{requires\_review}
7,Tipo de llamada vacío,180,\code{requires\_review}
8,Tipo de llamada \code{noise},866,se descarta
9,Par imposible corregido,23,\code{aa/hc} $\to$ \code{aa/hm}


In [6]:
# Tabla: cuántos valores distintos quedan en cada columna manual tras cada etapa.
def unique_values(series: pd.Series) -> int:
    return int(series.dropna().map(str).nunique())


def slugged(series: pd.Series) -> pd.Series:
    return series.map(lambda v: slugify(v, separator="_") if isinstance(v, str) else None)


n_species_dirs = len({species_of(p) for p in paired.values()})
etapas = pd.DataFrame(
    {
        "etapa": ["crudo", "tipografía", "sinónimos", "vocabulario"],
        "especie": [
            unique_values(raw_manual[RAW_SPECIES]),
            unique_values(slugged(raw_manual[RAW_SPECIES])),
            n_species_dirs,
            n_species_dirs,
        ],
        "llamada": [
            unique_values(raw_manual[RAW_CALL]),
            unique_values(slugged(raw_manual[RAW_CALL])),
            unique_values(audited[RAW_CALL]),
            unique_values(audited.loc[~audited["requires_review"], RAW_CALL]),
        ],
    }
)
write_rows("etapas", etapas.to_numpy().tolist())
etapas

,etapa,especie,llamada
0,crudo,19,63
1,tipografía,15,55
2,sinónimos,8,42
3,vocabulario,8,31


In [7]:
# Tablas: el vocabulario cerrado, el mapa de sinónimos y los pares marcados para revisión.
def readable(code: str) -> str:
    return code.replace("_", " ")


write_rows(
    "vocabulario",
    [
        (
            species.name,
            len(codes),
            ", ".join(f"\\code{{{c}}} ({readable(n)})" for c, n in codes.items()),
        )
        for species, codes in CALL_TYPES.items()
    ],
)
write_rows(
    "sinonimos",
    [
        (f"\\code{{{k.replace('_', '\\_')}}}", f"\\code{{{v}}}")
        for k, v in MANUAL_SYNONYMS.items()
        if k != "avevoc"
    ],
)
review = (
    kept.loc[kept["requires_review"]]
    .assign(call_type=kept[RAW_CALL].fillna(""))
    .groupby([RAW_SPECIES, RAW_CALL])
    .size()
    .sort_values(ascending=False)
)
n_vocab = sum(len(codes) for codes in CALL_TYPES.values())
headers = {k[4:]: v for k, v in counts.items() if k.startswith("col:")}

VALORES |= {
    "n_recordings_raw": number(len(recordings_raw)),
    "n_tables": number(len(tables)),
    "n_paired": number(len(paired)),
    "n_orphans": number(len(orphans)),
    "n_failed": number(len(failed)),
    "n_headers": number(len(headers)),
    "n_raw_rows": number(counts["filas"]),
    "n_kept": number(len(kept)),
    "pct_kept": percent(len(kept) / counts["filas"]),
    "n_typography": number(counts["tipografía"]),
    "n_synonyms": number(counts["sinónimo"]),
    "n_synonyms_map": number(len(MANUAL_SYNONYMS) - 1),
    "n_species_empty": number(counts["especie vacía"]),
    "n_species_mismatch": number(counts["especie≠carpeta"]),
    "n_noise": number(counts["ruido"]),
    "n_empty_call": number(counts["vacío"]),
    "n_fixed": number(counts["par imposible"]),
    "n_nyquist": number(counts["sobre Nyquist"]),
    "n_degenerate": number(counts["degenerada"]),
    "n_vocab": number(n_vocab),
    "n_review_audit": number(n_review),
    "n_review_vocab": number(counts["fuera de vocabulario"]),
    "n_review_groups": number(len(review)),
    "n_copies": number(n_copies),
    "n_duplicated_rows": number(n_duplicated_rows),
    "n_unique_recordings": number(len(groups)),
    "n_raw_call_values": number(int(etapas.llamada[0])),
    "n_raw_species_values": number(int(etapas.especie[0])),
    "n_vocab_call_values": number(int(etapas.llamada[3])),
}
review

species  call_type
as       cp           91
         pp           85
         ip           73
                      53
sm                    47
ac       bc           34
pt                    30
as       hc           25
ac                    21
lw       a            20
         b            15
as       bc           15
lw                    15
sb       ppc          11
                      11
sm       cc            9
sb       pcs           8
sm       hc            8
ac       chc           7
         cc            6
aa       sqc           6
sm       sc            5
pt       sqc           5
sm       sic           4
as       cc            4
         pc            3
pt       dc            3
sm       chc           3
sb       phc           3
lw       cs            3
         c             3
aa       hm            2
         gc            2
cc                     2
as       chc           1
aa                     1
         hf            1
pt       chc           1
lw       t             1
      

## 2. RE1.2 — selección de clases, partición y ventaneo

Lo que hay en `data/processed/`: se lee el caché tal como lo dejó `src/prepare_data.py` y se
comprueba que las particiones no comparten ninguna grabación.

In [8]:
from data import cache  # noqa: E402
from data.annotations import load_annotations  # noqa: E402
from prepare_data import EMPTY_RATIO, SPLIT_RATIOS  # noqa: E402
from utils.audio import hz_to_y, load_clip, mel_spectrogram, mel_to_db  # noqa: E402

SPLIT_NAMES = {"train": "Entrenamiento", "val": "Validación", "test": "Prueba"}
EDGE = 1e-3  # una caja toca el borde temporal si su lado queda a menos de esto

# Los parámetros de la selección se leen de `meta.json`, no del código de hoy: el caché en
# disco es el medio de verificación de RE1.2 y el conjunto con el que se entrenó OE2, y
# `meta.json` es justo el registro de cómo se construyó.
meta = json.loads((PROCESSED_DIR / "meta.json").read_text())
JOINED, EXCLUDED, UMBRAL = (
    meta["joined_labels"],
    list(meta["excluded_labels"]),
    int(meta["min_pair_count"]),
)
CLASES = list(json.loads((PROCESSED_DIR / "labels.json").read_text()).values())
INDICE_CLASE = {name: i for i, name in enumerate(CLASES)}

annotations = load_annotations()  # cleaned/, con la bandera `requires_review` que dejó RE1.1
# El pipeline recorta la frecuencia inferior al piso del mel antes de usar la caja.
annotations["low_freq_hz"] = annotations["low_freq_hz"].clip(lower=P.f_min)
annotations["bandwidth_hz"] = annotations["high_freq_hz"] - annotations["low_freq_hz"]
annotations["label"] = (annotations["species"] + "/" + annotations["call_type"]).replace(JOINED)
annotations["in_vocabulary"] = (
    pd.Series(list(zip(annotations["species"], annotations["call_type"], strict=True)))
    .isin(VALID_PAIRS)
    .to_numpy()
)
elegibles = annotations.loc[~annotations["requires_review"]]

por_par = annotations["label"].value_counts()
elegibles_por_par = elegibles["label"].value_counts().reindex(por_par.index).fillna(0).astype(int)
selection = pd.DataFrame({"anotaciones": por_par, "elegibles": elegibles_por_par})
selection["destino"] = np.select(
    [
        selection.index.isin(CLASES),
        selection.index.isin(EXCLUDED),
        selection["elegibles"].eq(0),
    ],
    ["clase", "excluida (frase)", "marcada para revisión"],
    default=f"$<{UMBRAL}$ anotaciones",
)
experiment = elegibles.loc[elegibles["label"].isin(CLASES)]
experiment = experiment.assign(
    recording=[str(Path(p).relative_to(RAW_DIR)) for p in experiment["audio_path"]]
)
pair_counts = annotations.loc[annotations["in_vocabulary"], "label"].value_counts()

# Volver a aplicar hoy las mismas reglas sobre el cleaned/ de hoy: si da otro conjunto de
# clases, el caché quedó atrás y el capítulo lo declara en lugar de taparlo.
hoy = elegibles["label"].value_counts()
CLASES_HOY = sorted(hoy[(hoy >= UMBRAL) & (~hoy.index.isin(EXCLUDED))].index)
desfase = sorted(set(CLASES) - set(CLASES_HOY))
print(
    f"{len(annotations)} anotaciones en cleaned/ · {int(annotations.requires_review.sum())} "
    f"marcadas para revisión · {len(pair_counts)} pares en vocabulario\n"
    f"caché: {len(CLASES)} clases con {len(experiment)} anotaciones\n"
    f"mismas reglas sobre el cleaned/ de hoy: {len(CLASES_HOY)} clases · "
    f"fuera ahora: {desfase or 'ninguna'}"
)
selection["destino"].value_counts()

19211 anotaciones en cleaned/ · 643 marcadas para revisión · 37 pares en vocabulario
caché: 25 clases con 17495 anotaciones
mismas reglas sobre el cleaned/ de hoy: 25 clases · fuera ahora: ninguna


destino
marcada para revisión    31
clase                    25
$<100$ anotaciones       10
excluida (frase)          3
Name: count, dtype: int64

In [9]:
# Los tres .pt se abren con mmap: cajas y etiquetas sin traer los mel a memoria.
def read_split(name: str) -> dict:
    return torch.load(cache.split_path(name), map_location="cpu", mmap=True, weights_only=True)


splits = {name: read_split(name) for name in cache.SPLITS}
sources = {name: cache.sources(name, len(data["images"])) for name, data in splits.items()}
recordings = {
    name: {str(Path(p).relative_to(RAW_DIR)) for p in found.recordings}
    for name, found in sources.items()
}
overlap = {
    f"{a} ∩ {b}": len(recordings[a] & recordings[b])
    for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
}
assert not any(overlap.values()), overlap  # la verificación de RE1.2: ninguna grabación se repite
shapes = {name: tuple(data["images"].shape) for name, data in splits.items()}
n_boxes = {name: [len(b) for b in data["boxes"]] for name, data in splits.items()}
composition = pd.DataFrame(
    {
        "grabaciones": {name: len(r) for name, r in recordings.items()},
        "ventanas": {name: len(n) for name, n in n_boxes.items()},
        "ventanas vacías": {name: sum(1 for n in ns if n == 0) for name, ns in n_boxes.items()},
        "cajas": {name: sum(ns) for name, ns in n_boxes.items()},
        "GB": {name: cache.split_path(name).stat().st_size / 1024**3 for name in splits},
    }
)
composition.loc["total"] = composition.sum()
write_rows(
    "particiones",
    [
        (
            SPLIT_NAMES.get(str(name), "\\textbf{Total}"),
            number(row.grabaciones),
            number(row.ventanas),
            number(row["ventanas vacías"]),
            percent(row["ventanas vacías"] / row.ventanas),
            number(row.cajas),
            number(row.GB, 2),
        )
        for name, row in composition.iterrows()
    ],
)
print(overlap, shapes)
composition.round(2)

{'train ∩ val': 0, 'train ∩ test': 0, 'val ∩ test': 0} {'train': (21751, 1, 128, 331), 'val': (9364, 1, 128, 331), 'test': (7598, 1, 128, 331)}


,grabaciones,ventanas,ventanas vacías,cajas,GB
train,1248.0,21751.0,4608.0,26938.0,3.44
val,690.0,9364.0,2713.0,10114.0,1.48
test,601.0,7598.0,2357.0,7931.0,1.20
total,2539.0,38713.0,9678.0,44983.0,6.13


In [10]:
# Por clase: anotaciones (cleaned/) y cajas por partición (ventanas del caché).
split_of = {r: name for name, rs in recordings.items() for r in rs}
experiment["split"] = experiment["recording"].map(split_of)
boxes_by_class = (
    pd.DataFrame(
        {
            name: pd.Series(torch.cat(data["labels"]).numpy()).value_counts()
            for name, data in splits.items()
        }
    )
    .fillna(0)
    .astype(int)
)
boxes_by_class.index = pd.Index([CLASES[i] for i in boxes_by_class.index])
per_class = pd.DataFrame(
    {
        "anotaciones": experiment["label"].value_counts(),
        "grabaciones": experiment.groupby("label")["recording"].nunique(),
    }
).join(boxes_by_class[list(cache.SPLITS)], how="outer").fillna(0).astype(int)
per_class["cajas"] = per_class[list(cache.SPLITS)].sum(axis=1)
per_class["cajas/anot."] = per_class["cajas"] / per_class["anotaciones"]
per_class["train %"] = per_class["train"] / per_class["cajas"]
per_class = per_class.sort_values("anotaciones", ascending=False)

nombre_legible = {
    f"{species.name.lower()}/{code}": name.replace("_", " ")
    for species, codes in CALL_TYPES.items()
    for code, name in codes.items()
}
nombre_legible["lw/trino"] = "trino (tr, tj, tt, tf)"
nombre_legible["sb/ppc"] = "play peep call (ppc, lpc)"
write_rows(
    "clases",
    [
        (
            f"\\code{{{name}}}" + ("\\textsuperscript{*}" if row.anotaciones == 0 else ""),
            nombre_legible[str(name)],
            number(row.anotaciones),
            number(row.grabaciones),
            number(row.train),
            number(row.val),
            number(row.test),
            percent(row["train %"], 0),
        )
        for name, row in per_class.iterrows()
    ]
    + [
        (
            "\\textbf{Total}",
            "",
            number(per_class.anotaciones.sum()),
            number(experiment["recording"].nunique()),
            number(per_class.train.sum()),
            number(per_class.val.sum()),
            number(per_class.test.sum()),
            percent(per_class.train.sum() / per_class.cajas.sum(), 0),
        )
    ],
)
# Pares que cleaned/ trae y el vocabulario no reconoce: quedan con la bandera de revisión.
fuera = (
    annotations.loc[~annotations["in_vocabulary"], "label"].value_counts().sort_values(
        ascending=False
    )
)
celdas = [(f"\\code{{{par}}}", number(n)) for par, n in fuera.items()]
mitad = (len(celdas) + 1) // 2  # la tabla va a dos columnas de pares
celdas += [("", "")] * (2 * mitad - len(celdas))
write_rows(
    "fuera_vocabulario",
    [a + b for a, b in zip(celdas[:mitad], celdas[mitad:], strict=True)],
)

# Pares del vocabulario que no entran: por anidamiento o por quedar bajo el umbral.
left_out = selection.loc[selection["destino"] != "clase"].sort_values(
    "anotaciones", ascending=False
)
write_rows(
    "excluidas",
    [
        (
            f"\\code{{{name}}}",
            nombre_legible.get(str(name), ""),
            number(row.anotaciones),
            row.destino,
        )
        for name, row in left_out.iterrows()
    ],
)
per_class.round(2)

,anotaciones,grabaciones,train,val,test,cajas,cajas/anot.,train %
label,,,,,,,,
ac/bc,2453,137,2838,1065,828,4731,1.93,0.60
lw/cs,2279,248,2667,1000,779,4446,1.95,0.60
sm/cc,2031,466,2353,883,686,3922,1.93,0.60
as/bc,1894,236,2216,831,647,3694,1.95,0.60
as/hc,1375,697,6933,2599,2021,11553,8.40,0.60
sb/ppc,1189,191,1340,529,464,2333,1.96,0.57
aa/gc,797,58,922,345,269,1536,1.93,0.60
sb/spc,729,166,859,323,250,1432,1.96,0.60
sm/fs,691,89,799,301,232,1332,1.93,0.60


In [11]:
# Qué le cuesta el ventaneo a cada clase: cajas por anotación, y cuántas cajas tocan un borde
# temporal de la ventana (la llamada sigue fuera de ella) o la ocupan entera.
all_boxes = torch.cat([b for data in splits.values() for b in data["boxes"] if len(b)])
all_labels = torch.cat([lab for data in splits.values() for lab in data["labels"]])
at_edge = (all_boxes[:, 0] - all_boxes[:, 2] / 2 <= EDGE) | (
    all_boxes[:, 0] + all_boxes[:, 2] / 2 >= 1 - EDGE
)
full_width = all_boxes[:, 2] >= 1 - 2 * EDGE
windowing = pd.DataFrame(
    {
        "anotaciones": per_class["anotaciones"],
        "cajas": per_class["cajas"],
        "cajas por anotación": per_class["cajas/anot."],
        "> 3 s": experiment.groupby("label")["duration_s"].apply(
            lambda d: (d > P.clip_len_s).mean()
        ),
        "en el borde": pd.Series(at_edge.numpy())
        .groupby(all_labels.numpy())
        .mean()
        .rename(lambda i: CLASES[i]),
        "ventana entera": pd.Series(full_width.numpy())
        .groupby(all_labels.numpy())
        .mean()
        .rename(lambda i: CLASES[i]),
    }
).sort_values("cajas por anotación", ascending=False)
long_classes = windowing.index[windowing["> 3 s"] > 0.5].tolist()
rest = windowing.drop(index=long_classes)


def fila(name, r) -> tuple:
    return (
        name,
        number(r.anotaciones),
        number(r.cajas),
        number(r["cajas por anotación"], 1),
        percent(r["> 3 s"]),
        percent(r["en el borde"]),
        percent(r["ventana entera"]),
    )


def agregado(frame: pd.DataFrame) -> pd.Series:
    peso_anot, peso_cajas = frame.anotaciones, frame.cajas
    return pd.Series(
        {
            "anotaciones": peso_anot.sum(),
            "cajas": peso_cajas.sum(),
            "cajas por anotación": peso_cajas.sum() / peso_anot.sum(),
            "> 3 s": (frame["> 3 s"] * peso_anot).sum() / peso_anot.sum(),
            "en el borde": (frame["en el borde"] * peso_cajas).sum() / peso_cajas.sum(),
            "ventana entera": (frame["ventana entera"] * peso_cajas).sum() / peso_cajas.sum(),
        }
    )


write_rows(
    "ventaneo",
    [fila(f"\\code{{{n}}}", r) for n, r in windowing.loc[long_classes].iterrows()]
    + [
        fila(f"Resto ({len(rest)} clases)", agregado(rest)),
        fila("\\textbf{Total}", agregado(windowing)),
    ],
)
# Histograma de cajas por ventana: pocas barras, va a pgfplots.
per_window = pd.Series([n for ns in n_boxes.values() for n in ns if n > 0])
write_data(
    "cajas_por_ventana",
    per_window.value_counts().sort_index().rename_axis("cajas").rename("ventanas").reset_index(),
)
print(f"mediana {per_window.median():.0f} · máximo {per_window.max()}")
windowing.round(3)

mediana 1 · máximo 11


,anotaciones,cajas,cajas por anotación,> 3 s,en el borde,ventana entera
as/hc,1375,11553,8.402,0.947,0.996,0.775
pt/dc,632,3240,5.127,0.910,0.992,0.635
ac/gc,140,318,2.271,0.171,0.557,0.138
ac/chc,229,468,2.044,0.035,0.421,0.009
sb/sc,105,209,1.990,0.000,0.172,0.000
sm/sc,163,324,1.988,0.037,0.256,0.015
ac/sc,158,313,1.981,0.019,0.438,0.019
pt/sqc,387,764,1.974,0.000,0.079,0.000
lw/sqc,154,303,1.968,0.000,0.314,0.000
sb/spc,729,1432,1.964,0.000,0.055,0.000


In [12]:
# Tres ventanas de entrenamiento tal como las recibe el modelo: la más densa, una llamada larga
# cortada por los dos bordes y una vacía. Son imágenes: matplotlib.
train = splits["train"]
train_counts = np.array(n_boxes["train"])
densest = int(train_counts.argmax())
long_label = INDICE_CLASE[long_classes[0]]
long_window = next(
    i
    for i, (b, lab) in enumerate(zip(train["boxes"], train["labels"], strict=True))
    if len(b) == 1 and int(lab[0]) == long_label and float(b[0, 2]) >= 1 - 2 * EDGE
)
empty_window = int(np.flatnonzero(train_counts == 0)[0])

for index, name in (
    (densest, "ventana_densa"),
    (long_window, "ventana_larga"),
    (empty_window, "ventana_vacia"),
):
    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN / 3, 2.2), constrained_layout=True)
    ax.imshow(
        mel_to_db(train["images"][index, 0]).numpy(),
        origin="lower",
        aspect="auto",
        cmap="magma",
        extent=(0, 1, 0, 1),
    )
    vistas: set[int] = set()  # once cajas de la misma clase no se rotulan once veces
    for (cx, cy, w, h), lab in zip(
        train["boxes"][index].tolist(), train["labels"][index].tolist(), strict=True
    ):
        ax.add_patch(
            Rectangle((cx - w / 2, cy - h / 2), w, h, fill=False, edgecolor=VERDE, linewidth=1.2)
        )
        if lab not in vistas:
            vistas.add(lab)
            ax.text(
                cx - w / 2,
                min(cy + h / 2, 0.96),
                CLASES[lab],
                color=VERDE,
                fontsize=6.5,
                va="bottom",
            )
    found = sources["train"]
    recording = Path(found.recordings[found.recording_of_window[index]]).name
    ax.set_title(f"{recording} · {found.clip_start_s[index]:.1f} s", fontsize=6.5)
    ax.set(
        xlabel="tiempo (fracción)",
        ylabel="frecuencia (mel)",
        xticks=[0, 0.5, 1],
        yticks=[0, 0.5, 1],
    )
    ax.grid(False)
    save(fig, name)
    plt.show()

In [13]:
shutil.copy(PROCESSED_DIR / "meta.json", FIG / "meta.json")


def total(column: str, split: str = "total") -> float:
    return scalar(composition, split, column)


VALORES |= {
    "n_annotations_cleaned": number(len(annotations)),
    "n_vocab_pairs": number(len(pair_counts)),
    "n_annotations_vocab": number(int(pair_counts.sum())),
    "n_pairs_cleaned": number(int(annotations["label"].nunique())),
    "n_pairs_out_vocab": number(int(len(fuera))),
    "n_annotations_out_vocab": number(int((~annotations["in_vocabulary"]).sum())),
    "n_review": number(int(annotations["requires_review"].sum())),
    "pct_review": percent(float(annotations["requires_review"].mean())),
    "n_review_pairs": number(int(annotations.loc[annotations["requires_review"], "label"].nunique())),
    "n_classes_today": number(len(CLASES_HOY)),
    "classes_drift": ", ".join(f"\\code{{{c}}}" for c in desfase) or "ninguna",
    "n_cleaned_delta": number(abs(len(kept) - len(annotations))),
    "n_classes": number(len(CLASES)),
    "n_experiment": number(len(experiment)),
    "pct_experiment": percent(len(experiment) / int(pair_counts.sum())),
    "n_left_out": number(len(left_out)),
    "n_left_out_annotations": number(int(left_out.anotaciones.sum())),
    "n_below_threshold": number(int((left_out.destino != "excluida (frase)").sum())),
    "n_excluded_phrase": number(int((left_out.destino == "excluida (frase)").sum())),
    "n_recordings_experiment": number(experiment["recording"].nunique()),
    "n_windows": number(total("ventanas")),
    "n_boxes": number(total("cajas")),
    "n_empty": number(total("ventanas vacías")),
    "pct_empty": percent(total("ventanas vacías") / total("ventanas")),
    "n_windows_train": number(total("ventanas", "train")),
    "n_windows_val": number(total("ventanas", "val")),
    "n_windows_test": number(total("ventanas", "test")),
    "n_boxes_train": number(total("cajas", "train")),
    "n_boxes_val": number(total("cajas", "val")),
    "n_boxes_test": number(total("cajas", "test")),
    "gb_total": number(total("GB"), 1),
    "n_frames": number(P.n_frames),
    "n_mels": number(P.n_mels),
    "boxes_per_annotation": number(float(windowing.cajas.sum() / windowing.anotaciones.sum()), 1),
    "long_classes": " y ".join(f"\\code{{{c}}}" for c in long_classes),
    "pct_long_boxes": percent(
        windowing.loc[long_classes, "cajas"].sum() / windowing.cajas.sum(), 0
    ),
    "long_first": f"\\code{{{long_classes[0]}}}",
    "long_first_boxes_per": number(scalar(windowing, long_classes[0], "cajas por anotación"), 1),
    "long_first_edge": percent(scalar(windowing, long_classes[0], "en el borde")),
    "long_first_full": percent(scalar(windowing, long_classes[0], "ventana entera")),
    "median_per_window": number(float(per_window.median())),
    "max_per_window": number(int(per_window.max())),
    "pct_leq_four": percent((per_window <= 4).mean(), 0),
    "largest_class": f"\\code{{{per_class.index[0]}}}",
    "largest_class_train": number(int(per_class.train.iloc[0])),
    "smallest_class": f"\\code{{{per_class.index[-1]}}}",
    "smallest_class_train": number(int(per_class.train.iloc[-1])),
    "smallest_class_n": number(int(per_class.anotaciones.iloc[-1])),
    "n_small_classes": number(int((per_class.train < 300).sum())),
    "class_imbalance": number(
        float(per_class.train.iloc[0] / max(per_class.train.iloc[-1], 1)), 0
    ),
    "db_low": number(meta["db_range"][0], 1),
    "db_high": number(meta["db_range"][1], 1),
    "split_ratios": " / ".join(number(100 * r, 1) for r in SPLIT_RATIOS),
    "empty_ratio": percent(EMPTY_RATIO, 0),
    "min_pair_count": number(UMBRAL),
    "seed": number(meta["seed"]),
    "clip_len": number(P.clip_len_s, 0),
    "clip_hop": number(P.clip_hop_s, 1),
}
sorted(p.name for p in FIG.iterdir())

['anidamiento_frase.png',
 'anotaciones_por_par.png',
 'audio_corrupcion_leve.png',
 'audio_corrupcion_severa.png',
 'aviso_audio_danado.png',
 'cajas_por_clase_particion.png',
 'data',
 'etiquetas_especie_crudas.png',
 'etiquetas_llamada_crudas.png',
 'formas_ac_bc.png',
 'formas_sm_fs.png',
 'galeria_aa_gc.png',
 'galeria_ac_bc.png',
 'galeria_as_bc.png',
 'galeria_as_hc.png',
 'galeria_lw_cs.png',
 'galeria_pt_dc.png',
 'galeria_sb_ppc.png',
 'galeria_sb_spc.png',
 'galeria_sm_cc.png',
 'geometria_facetas.png',
 'meta.json',
 'raven_anotando.png',
 'tabla_raven_cruda.png',
 'tablas',
 'valores.tex',
 'ventana_densa.png',
 'ventana_larga.png',
 'ventana_vacia.png']

## 3. RE1.3 — composición, geometría y galería

Las cifras de la ficha: qué aporta cada especie, cómo es la geometría de cada clase y cómo se
ven las llamadas sobre el espectrograma.

In [14]:
from data.manifest import duration_of  # noqa: E402

CIENTIFICO = {
    "aa": "Aotus azarae",
    "ac": "Ateles chamek",
    "as": "Alouatta sara",
    "cc": "Cebus cuscinus",
    "lw": "Leontocebus weddelli",
    "pt": "Plecturocebus toppini",
    "sb": "Saimiri boliviensis",
    "sm": "Sapajus macrocephalus",
}
COLOR_ESPECIE = {
    "aa": "#7c3aed",
    "ac": AMBAR,
    "as": "#dc2626",
    "cc": GRIS,
    "lw": AZUL,
    "pt": "#0d9488",
    "sb": CELESTE,
    "sm": VERDE,
}
# Clases cuya nube duración × ancho de banda se parte en dos (se ve en la figura de formas).
FORMAS_PARTIDAS = ("ac/bc", "sm/fs")

# Una grabación dañada no se puede abrir: cuenta como 0 s (tampoco entra al caché).
durations = pd.Series(
    {path: duration_of(path) or 0.0 for path in annotations["audio_path"].unique()}
)
print(f"{len(durations)} grabaciones · {durations.sum() / 3600:.1f} h de audio anotado")

2691 grabaciones · 25.0 h de audio anotado


In [15]:
# Composición por especie: grabaciones, horas, anotaciones y qué parte entra al experimento.
por_especie = (
    pd.DataFrame(
        {
            "grabaciones": annotations.groupby("species")["audio_path"].nunique(),
            "horas": annotations.groupby("species").apply(
                lambda g: durations[g["audio_path"].unique()].sum() / 3600, include_groups=False
            ),
            "anotaciones": annotations.groupby("species").size(),
            "pares": annotations[annotations["in_vocabulary"]]
            .groupby("species")["label"]
            .nunique(),
            "clases": experiment.groupby("species")["label"].nunique(),
            "en experimento": experiment.groupby("species").size(),
        }
    )
    .fillna(0)
    .astype({"clases": int, "en experimento": int, "pares": int})
    .sort_values("anotaciones", ascending=False)
)
write_rows(
    "especies",
    [
        (
            f"\\textit{{{CIENTIFICO[str(sp)]}}}",
            str(sp).upper(),
            number(row.grabaciones),
            number(row.horas, 1),
            number(row.anotaciones),
            number(row.pares),
            number(row.clases),
            number(row["en experimento"]),
        )
        for sp, row in por_especie.iterrows()
    ]
    + [
        (
            "\\textbf{Total}",
            "",
            number(len(durations)),
            number(durations.sum() / 3600, 1),
            number(len(annotations)),
            number(int(por_especie.pares.sum())),
            number(len(CLASES)),
            number(len(experiment)),
        )
    ],
)
por_especie.round(1)

,grabaciones,horas,anotaciones,pares,clases,en experimento
species,,,,,,
lw,410,3.7,4159,8,5,3450
sm,582,5.3,3668,8,5,3381
as,778,7.2,3619,2,2,3269
ac,384,3.7,3106,6,4,2980
sb,256,2.8,2348,4,4,2313
pt,320,2.8,1185,5,2,1019
aa,82,0.8,1095,3,3,1083
cc,7,0.0,31,1,0,0


In [16]:
# Geometría por clase: duración (mediana y cuartiles), banda y ancho de banda medianos.
grouped = experiment.groupby("label")
geometry = pd.DataFrame(
    {
        "n": grouped.size(),
        "dur_med": grouped["duration_s"].median(),
        "dur_q1": grouped["duration_s"].quantile(0.25),
        "dur_q3": grouped["duration_s"].quantile(0.75),
        "low": grouped["low_freq_hz"].median(),
        "high": grouped["high_freq_hz"].median(),
        "bw": grouped["bandwidth_hz"].median(),
        "bw_cv": grouped["bandwidth_hz"].std() / grouped["bandwidth_hz"].mean(),
    }
).sort_values("dur_med", ascending=False)


def segundos(value: float) -> str:
    return number(value, 2) if value < 1 else number(value, 1)


write_rows(
    "geometria",
    [
        (
            f"\\code{{{name}}}",
            number(r.n),
            f"{segundos(r.dur_med)} [{segundos(r.dur_q1)}--{segundos(r.dur_q3)}]",
            f"{number(r.low)}--{number(r.high)}",
            number(r.bw),
            number(r.bw_cv, 2),
        )
        for name, r in geometry.iterrows()
    ],
)
geometry.round(3)

,n,dur_med,dur_q1,dur_q3,low,high,bw,bw_cv
label,,,,,,,,
as/hc,1375,10.993,7.129,15.807,47.385,768.795,717.500,0.229
pt/dc,632,6.671,4.756,9.549,25.000,1474.273,1385.174,0.680
ac/chc,229,1.201,0.894,1.594,191.701,3377.900,3207.839,0.503
ac/sc,158,1.180,0.867,1.422,2640.291,10725.782,8134.951,0.345
ac/gc,140,1.012,0.657,1.922,197.538,1465.462,1275.916,0.548
lw/sqc,154,0.812,0.605,1.048,5033.828,12949.800,7753.000,0.480
sb/pcc,290,0.726,0.496,1.059,6535.596,11754.635,4955.498,0.339
sm/sc,163,0.586,0.355,0.822,3380.726,10752.900,6945.800,0.364
sm/pc,242,0.550,0.404,0.823,2805.884,4435.345,1541.894,0.424


In [17]:
# Las dos etiquetas que esconden dos formas: la nube se parte en dos. Son nubes de puntos:
# matplotlib, una imagen por clase del ancho de media página.
for name in FORMAS_PARTIDAS:
    subset = experiment.loc[experiment["label"] == name]
    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN / 2, 2.4), constrained_layout=True)
    ax.scatter(
        subset["duration_s"] * 1000,
        subset["bandwidth_hz"] / 1000,
        s=6,
        color=COLOR_ESPECIE[name.split("/")[0]],
        alpha=0.45,
        linewidths=0,
    )
    ax.set(
        xscale="log",
        xlim=(40, 2000),
        xlabel="duración (ms)",
        ylabel="ancho de banda (kHz)",
        title=f"{name}  (n = {len(subset)}, CV del ancho = {scalar(geometry, name, 'bw_cv'):.2f})",
    )
    ax.set_xticks([50, 100, 200, 500, 1000], ["50", "100", "200", "500", "1000"])
    ax.xaxis.set_minor_formatter(NullFormatter())
    save(fig, f"formas_{name.replace('/', '_')}")
    plt.show()

In [18]:
# Galería: la clase más anotada de cada especie más las siguientes en volumen hasta nueve; de
# cada una, la anotación de duración más cercana a la mediana de su clase, en su contexto.
COLUMNAS = 3


def ejemplo_mediano(name: str) -> pd.Series:
    subset = experiment.loc[experiment["label"] == name]
    return subset.loc[(subset["duration_s"] - subset["duration_s"].median()).abs().idxmin()]


by_size = experiment["label"].value_counts()
per_species = by_size.groupby(by_size.index.str.split("/").str[0]).idxmax().tolist()
elegidas = (
    per_species
    + [c for c in by_size.index if c not in per_species][: COLUMNAS**2 - len(per_species)]
)
elegidas = geometry.loc[elegidas].sort_values("low").index.tolist()

to_mel = mel_spectrogram()
frames_per_s = P.n_frames / P.clip_len_s
subfiguras: list[str] = []
for name in elegidas:
    fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN / 3, 1.75), constrained_layout=True)
    row = ejemplo_mediano(name)
    span = min(max(row["duration_s"] * 2.5, 0.5), P.clip_len_s)
    start = max(row["begin_time_s"] - (span - row["duration_s"]) / 2, 0.0)
    mel = mel_to_db(to_mel(load_clip(row["audio_path"], start)))[:, : int(span * frames_per_s)]
    ax.imshow(mel.numpy(), origin="lower", aspect="auto", cmap="magma", extent=(0, span, 0, 1))
    y0, y1 = hz_to_y(np.array([row["low_freq_hz"], row["high_freq_hz"]]), P)
    ax.add_patch(
        Rectangle(
            (row["begin_time_s"] - start, y0),
            row["duration_s"],
            y1 - y0,
            fill=False,
            edgecolor=VERDE,
            linewidth=1.3,
        )
    )
    ax.set_yticks(hz_to_y(np.array([1000.0, 5000.0, 15000.0]), P).tolist(), ["1k", "5k", "15k"])
    ax.set_xlabel("s", labelpad=1)
    ax.grid(False)
    stem = f"galeria_{name.replace('/', '_')}"
    save(fig, stem)
    plt.show()
    subfiguras.append(
        "\\begin{subfigure}[t]{0.32\\textwidth}\n"
        f"    \\includegraphics[width=\\linewidth]{{{RUTA_FIG}/{stem}}}\n"
        f"    \\caption{{\\code{{{name}}}, {nombre_legible[name]}, "
        f"{segundos(row['duration_s'])}\\,s}}\n"
        "\\end{subfigure}"
    )
# Tres por fila: `\hfill` entre vecinas y un salto al cerrar cada fila.
pegamento = [
    "\\\\[1.5ex]\n" if (i + 1) % COLUMNAS == 0 else "\\hfill\n" for i in range(len(subfiguras) - 1)
]
cuerpo = "".join(
    parte for par in zip(subfiguras, [*pegamento, "\n"], strict=True) for parte in par
)
(TAB / "galeria.tex").write_text("% generado por el cuaderno; no editar\n" + cuerpo)
print(f"galería: {', '.join(elegidas)}")

galería: pt/dc, as/hc, aa/gc, as/bc, sb/spc, ac/bc, sm/cc, sb/ppc, lw/cs


In [19]:
# Distribución temporal: las grabaciones se llaman AAAAMMDD_HHMMSS.wav (a veces AAMMDD_HHMM).
# Pocas barras y ejes con tipografía del documento: van a pgfplots desde los .dat.
nombres = pd.Series([Path(p).stem for p in durations.index]).str.extract(r"^(\d{6}|\d{8})_(\d{2})")
fechas = pd.to_datetime(
    nombres[0].where(nombres[0].str.len() == 8, "20" + nombres[0]), format="%Y%m%d", errors="coerce"
)
horas = nombres[1].dropna().astype(int).value_counts().sort_index()
write_data("horas", horas.rename_axis("hora").rename("grabaciones").reset_index())
meses = fechas.dt.to_period("M").value_counts().sort_index()
write_data(
    "meses", meses.rename_axis("mes").rename("grabaciones").reset_index().astype(str)
)
mes_pico = meses.idxmax()
extremos = {
    "duración": (experiment["duration_s"].min(), experiment["duration_s"].max()),
    "ancho de banda": (experiment["bandwidth_hz"].min(), experiment["bandwidth_hz"].max()),
}
VALORES |= {
    "n_recordings": number(len(durations)),
    "n_unreadable": number(int((durations == 0).sum())),
    "n_recordings_cache": number(int(total("grabaciones"))),
    "hours": number(durations.sum() / 3600, 1),
    "median_recording_s": number(durations.median()),
    "pct_short_recordings": percent((durations <= 60).mean(), 0),
    "n_annotations": number(len(annotations)),
    "n_species": number(annotations["species"].nunique()),
    "n_species_experiment": number(experiment["species"].nunique()),
    "dur_min_ms": number(extremos["duración"][0] * 1000),
    "dur_max_s": number(extremos["duración"][1], 1),
    "dur_ratio": number(extremos["duración"][1] / extremos["duración"][0]),
    "bw_min": number(extremos["ancho de banda"][0]),
    "bw_max": number(extremos["ancho de banda"][1]),
    "bw_ratio": number(extremos["ancho de banda"][1] / extremos["ancho de banda"][0]),
    "top_ten_share": percent(pair_counts.head(10).sum() / pair_counts.sum(), 0),
    "largest_pair": f"\\code{{{pair_counts.index[0]}}}",
    "largest_pair_n": number(pair_counts.iloc[0]),
    "split_shapes": " y ".join(f"\\code{{{c}}}" for c in FORMAS_PARTIDAS),
    "split_shapes_cv": " y ".join(
        number(scalar(geometry, c, "bw_cv"), 2) for c in FORMAS_PARTIDAS
    ),
    "first_date": fechas.min().strftime("%d/%m/%Y"),
    "last_date": fechas.max().strftime("%d/%m/%Y"),
    "peak_month": f"{mes_pico.month:02d}/{mes_pico.year}",
    "longest_class": f"\\code{{{geometry.index[0]}}}",
    "longest_class_s": number(geometry["dur_med"].iloc[0], 1),
    "shortest_class": f"\\code{{{geometry.index[-1]}}}",
    "shortest_class_ms": number(geometry["dur_med"].iloc[-1] * 1000),
    "most_species_pairs": number(int(por_especie.pares.max())),
    "most_species": f"\\textit{{{CIENTIFICO[str(por_especie.pares.idxmax())]}}}",
    "gallery_classes": number(len(elegidas)),
}
horas.to_dict()

{0: 2,
 4: 5,
 5: 311,
 6: 481,
 7: 387,
 8: 229,
 9: 274,
 10: 243,
 11: 143,
 12: 115,
 13: 7,
 14: 35,
 15: 92,
 16: 224,
 17: 85,
 19: 24,
 20: 15,
 21: 5,
 23: 10}

## 4. Las cuatro figuras de datos del capítulo

Nubes de puntos, barras por clase y un espectrograma: demasiados puntos o demasiadas
categorías para pgfplots, y una de ellas es una imagen. Van en matplotlib.

In [20]:
# Anotaciones por par, con el umbral del experimento sobre la distribución completa.
destino_de = selection["destino"].to_dict()
todos = annotations["label"].value_counts()
en_vocabulario = dict(
    zip(annotations["label"], annotations["in_vocabulary"], strict=True)
)
COLOR_DESTINO = {
    "clase": AZUL,
    "fuera de vocabulario": AMBAR,
    "excluida (frase)": "#7c3aed",
    f"$<{UMBRAL}$ anotaciones": GRIS,
    "marcada para revisión": ROSA,
}
ETIQUETA_DESTINO = {
    "clase": f"clase del experimento ({len(CLASES)})",
    "fuera de vocabulario": "fuera de vocabulario",
    "excluida (frase)": "excluida por anidamiento",
    f"$<{UMBRAL}$ anotaciones": f"bajo el umbral de {UMBRAL}",
    "marcada para revisión": "marcada para revisión (RE1.1)",
}
grupos = [
    "fuera de vocabulario" if not en_vocabulario.get(str(p), False)
    else destino_de.get(str(p), f"$<{UMBRAL}$ anotaciones")
    for p in todos.index
]
# A dos columnas: 57 pares en una sola columna obligan a encoger la figura y los rótulos
# dejan de leerse.
mitad_pares = (len(todos) + 1) // 2
fig, axes = plt.subplots(
    1, 2, figsize=(TEXT_WIDTH_IN, 0.125 * mitad_pares + 1.0), sharex=True
)
for ax, desde in zip(axes, (0, mitad_pares), strict=True):
    bloque = todos.iloc[desde : desde + mitad_pares]
    grupos_bloque = grupos[desde : desde + mitad_pares]
    ys = np.arange(len(bloque))
    for y, valor, grupo in zip(ys, bloque.to_numpy(), grupos_bloque, strict=True):
        color = COLOR_DESTINO[grupo]
        ax.plot([1, valor], [y, y], color="#e3e3e3", linewidth=0.8, zorder=1)
        ax.scatter([valor], [y], s=14, color=color, zorder=3)
        ax.text(
            valor * 1.2, y, f"{valor:,}".replace(",", " "), va="center", fontsize=6, color=color
        )
    ax.axvline(UMBRAL, color="#111111", linewidth=0.9, linestyle=(0, (4, 3)), zorder=2)
    ax.set(xscale="log", xlim=(1, float(todos.max()) * 3.4), ylim=(len(bloque) - 0.4, -1.0))
    ax.set_yticks(ys, bloque.index, fontsize=6.4)
    for etiqueta, grupo in zip(ax.get_yticklabels(), grupos_bloque, strict=True):
        etiqueta.set_color(COLOR_DESTINO[grupo])
    ax.set_xlabel("anotaciones (escala logarítmica)", fontsize=7)
    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(False, axis="y")
axes[0].text(UMBRAL * 1.25, -0.7, f"umbral {UMBRAL}", fontsize=6.5, va="center", color="#111")
vistos = list(dict.fromkeys(grupos))
# Arriba a la derecha del panel derecho: es la única zona sin un solo punto, porque ahí
# ningún par de la segunda mitad llega a 100 anotaciones.
axes[1].legend(
    handles=[
        plt.Line2D(
            [], [], marker="o", linestyle="", color=COLOR_DESTINO[g], markersize=4,
            label=ETIQUETA_DESTINO[g],
        )
        for g in COLOR_DESTINO
        if g in vistos
    ],
    loc="upper right",
    fontsize=6,
)
fig.subplots_adjust(wspace=0.35)
save(fig, "anotaciones_por_par")
plt.show()

In [21]:
# Duración frente a ancho de banda, una faceta por clase: dónde una etiqueta esconde dos formas.
clases = geometry.index.tolist()
n_cols = 5
n_rows = int(np.ceil(len(clases) / n_cols))
fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(TEXT_WIDTH_IN, 1.35 * n_rows), sharex=True, sharey=True
)
for ax, name in zip(axes.flat, clases, strict=False):
    rows = experiment.loc[experiment["label"] == name]
    ax.scatter(
        experiment["duration_s"] * 1000,
        experiment["bandwidth_hz"] / 1000,
        s=1.5,
        color="#d9d9d9",
        linewidths=0,
        zorder=1,
    )
    ax.scatter(
        rows["duration_s"] * 1000,
        rows["bandwidth_hz"] / 1000,
        s=3,
        color=COLOR_ESPECIE[name.split("/")[0]],
        alpha=0.55,
        linewidths=0,
        zorder=2,
    )
    ax.set_xscale("log")
    ax.set_title(f"{name}  (n = {len(rows):,})".replace(",", " "), fontsize=6.2, loc="left", pad=2)
    ax.tick_params(labelsize=5.6)
    ax.grid(True, alpha=0.2)
for ax in axes.flat[len(clases) :]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("duración (ms)", fontsize=6.2)
for ax in axes[:, 0]:
    ax.set_ylabel("banda (kHz)", fontsize=6.2)
fig.subplots_adjust(hspace=0.55, wspace=0.12)
save(fig, "geometria_facetas")
plt.show()

In [22]:
# Cajas por clase y partición, con las anotaciones de la clase como referencia: la diferencia es
# lo que el ventaneo multiplica en las clases largas.
tabla = per_class.sort_values("cajas", ascending=False)
ys = np.arange(len(tabla))
fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 0.19 * len(tabla) + 1.0))
izquierda = np.zeros(len(tabla))
for name, color in (("train", AZUL), ("val", CELESTE), ("test", AMBAR)):
    ax.barh(ys, tabla[name].to_numpy(), height=0.66, left=izquierda, color=color, label=name)
    izquierda = izquierda + tabla[name].to_numpy()
ax.scatter(tabla["anotaciones"], ys, s=11, color="#111111", zorder=4, label="anotaciones")
for y, total_cajas in zip(ys, izquierda, strict=True):
    ax.text(
        total_cajas * 1.02,
        y,
        f"{int(total_cajas):,}".replace(",", " "),
        va="center",
        fontsize=5.8,
        color="#525252",
    )
ax.set(xlim=(0, izquierda.max() * 1.16), ylim=(len(tabla) - 0.4, -0.9))
ax.set_yticks(ys, tabla.index, fontsize=6.2)
ax.set_xlabel("cajas en el caché de ventanas")
ax.grid(True, axis="x", alpha=0.25)
ax.grid(False, axis="y")
ax.legend(loc="lower right", ncol=4, fontsize=6.2)
save(fig, "cajas_por_clase_particion")
plt.show()

In [23]:
# El anidamiento frase–sílaba sobre una grabación real: una frase con sus sílabas dentro.
import soundfile as sf  # noqa: E402
import torchaudio  # noqa: E402

PARES_ANIDADOS = [("lw/cc", "lw/cs"), ("sm/fc", "sm/fs"), ("sb/pcs", "sb/ppc")]


def cargar_segmento(audio_path, start_s: float, duration_s: float) -> torch.Tensor:
    with sf.SoundFile(str(audio_path)) as handle:
        handle.seek(int(start_s * handle.samplerate))
        frames = handle.read(int(duration_s * handle.samplerate), dtype="float32", always_2d=True)
        waveform = torch.from_numpy(frames.mean(axis=1))
        if handle.samplerate != P.target_sr:
            waveform = torchaudio.functional.resample(waveform, handle.samplerate, P.target_sr)
    return waveform


anidamiento = []
for frase_par, silaba_par in PARES_ANIDADOS:
    frases = annotations.loc[annotations["label"] == frase_par]
    silabas = annotations.loc[annotations["label"] == silaba_par]
    if frases.empty or silabas.empty:
        continue
    con_silaba = 0
    for audio_path, grupo in frases.groupby("audio_path"):
        dentro = silabas.loc[silabas["audio_path"] == audio_path]
        if dentro.empty:
            continue
        medio = ((dentro["begin_time_s"] + dentro["end_time_s"]) / 2).to_numpy()[None, :]
        inicio = grupo["begin_time_s"].to_numpy()[:, None]
        fin = grupo["end_time_s"].to_numpy()[:, None]
        con_silaba += int(((medio >= inicio) & (medio <= fin)).any(axis=1).sum())
    anidamiento.append(
        {"frase": frase_par, "silaba": silaba_par, "frases": len(frases),
         "con sílaba dentro": con_silaba / len(frases)}
    )
anidamiento = pd.DataFrame(anidamiento).set_index("frase")

frase_par, silaba_par = PARES_ANIDADOS[0]
mejor = None
for r in annotations.loc[annotations["label"] == frase_par].itertuples():
    contenidas = annotations.loc[
        (annotations["audio_path"] == r.audio_path)
        & (annotations["label"] == silaba_par)
        & (annotations["begin_time_s"] >= r.begin_time_s - 0.05)
        & (annotations["end_time_s"] <= r.end_time_s + 0.05)
    ]
    if mejor is None or len(contenidas) > len(mejor[1]):
        mejor = (r, contenidas)
frase, silabas = mejor

pad = max(frase.duration_s * 0.18, 0.25)
inicio = max(frase.begin_time_s - pad, 0.0)
duracion = frase.duration_s + 2 * pad
mel = mel_to_db(to_mel(cargar_segmento(frase.audio_path, inicio, duracion)))
fig, ax = plt.subplots(figsize=(TEXT_WIDTH_IN, 2.7), constrained_layout=True)
ax.imshow(
    mel.numpy(), origin="lower", aspect="auto", cmap="magma", extent=(0, duracion, 0, 1),
    vmin=float(np.percentile(mel.numpy(), 2)), vmax=float(np.percentile(mel.numpy(), 99.5)),
)
y0, y1 = hz_to_y(np.array([frase.low_freq_hz, frase.high_freq_hz]), P)
ax.add_patch(
    Rectangle(
        (frase.begin_time_s - inicio, y0), frase.duration_s, y1 - y0, fill=False,
        edgecolor=AMBAR, linewidth=1.8, linestyle=(0, (4, 2)),
    )
)
ax.text(
    frase.begin_time_s - inicio, y1 + 0.015,
    f"{frase_par} (frase, {frase.duration_s:.1f} s)", color=AMBAR, fontsize=7, va="bottom",
)
for r in silabas.itertuples():
    sy0, sy1 = hz_to_y(np.array([r.low_freq_hz, r.high_freq_hz]), P)
    ax.add_patch(
        Rectangle(
            (r.begin_time_s - inicio, sy0), r.duration_s, sy1 - sy0, fill=False,
            edgecolor=VERDE, linewidth=1.1,
        )
    )
ax.text(
    (frase.begin_time_s + frase.end_time_s) / 2 - inicio, 0.05,
    f"{len(silabas)} sílabas {silaba_par} dentro de una sola frase",
    ha="center", va="center", fontsize=7, color="#0b3d1f",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.9),
)
ax.set_yticks(
    hz_to_y(np.array([1000.0, 5000.0, 10000.0, 20000.0]), P).tolist(), ["1k", "5k", "10k", "20k"]
)
ax.set(xlabel="tiempo (s)", ylabel="frecuencia (Hz, eje mel)")
ax.grid(False)
save(fig, "anidamiento_frase")
plt.show()
anidamiento.round(3)

,silaba,frases,con sílaba dentro
frase,,,
lw/cc,lw/cs,536,0.942
sm/fc,sm/fs,126,0.968
sb/pcs,sb/ppc,8,0.000


In [24]:
VALORES |= {
    "nesting_phrase": f"\\code{{{frase_par}}}",
    "nesting_syllable": f"\\code{{{silaba_par}}}",
    "pct_nested_lw": percent(scalar(anidamiento, "lw/cc", "con sílaba dentro")),
    "pct_nested_sm": percent(scalar(anidamiento, "sm/fc", "con sílaba dentro")),
    "nesting_example_syllables": number(len(silabas)),
    "nesting_example_s": number(frase.duration_s, 1),
}
write_macros(VALORES)
print(f"{len(VALORES)} macros en {FIG / 'valores.tex'}")
print("\n".join(f"  {p.name}" for p in sorted(FIG.iterdir())))

123 macros en /home/fcandia/tesis-primate/research/capitulos/04_oe1_curacion/figures/valores.tex
  anidamiento_frase.png
  anotaciones_por_par.png
  audio_corrupcion_leve.png
  audio_corrupcion_severa.png
  aviso_audio_danado.png
  cajas_por_clase_particion.png
  data
  etiquetas_especie_crudas.png
  etiquetas_llamada_crudas.png
  formas_ac_bc.png
  formas_sm_fs.png
  galeria_aa_gc.png
  galeria_ac_bc.png
  galeria_as_bc.png
  galeria_as_hc.png
  galeria_lw_cs.png
  galeria_pt_dc.png
  galeria_sb_ppc.png
  galeria_sb_spc.png
  galeria_sm_cc.png
  geometria_facetas.png
  meta.json
  raven_anotando.png
  tabla_raven_cruda.png
  tablas
  valores.tex
  ventana_densa.png
  ventana_larga.png
  ventana_vacia.png
